# Silver Layer

## Sales Details
Information about customer, product bought, date ordered/shipped/due, as well the quantity, price and total sales per customer

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col, trim, upper, now, isnull, current_date, max, min, isnotnull, length, lag, date_add, lead, isnull, ifnull
from pyspark.sql.types import *

## Read table from bronze layer

In [0]:
df = spark.read.table("db_project.bronze.crm_sales_details")

# Transform data

## Clear strings

Each column with string datatype should be trimed from possible additional spaces

In [0]:
for field in df.schema.fields:
    # display(field.dataType)
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

Show first 100 rows, table ordered by customer id

In [0]:
df.orderBy("sls_cust_id").limit(100).display()

## Nulls within the table

Check for possible nulls within the table

In [0]:
tets_nulls = df.where(
                df.sls_cust_id.isNull() |
                df.sls_ord_num.isNull() |
                df.sls_prd_key.isNull() |
                df.sls_sales.isNull() |
                df.sls_quantity.isNull() |
                df.sls_price.isNull() |
                df.sls_order_dt.isNull()
            )
tets_nulls.display()

## Incorrect sales values

Check if sales are within the norm of business idea of sales = qty * price

In [0]:
test_sales = df.where(df.sls_sales != df.sls_quantity*df.sls_price)
test_sales.display()

Sales and prices appear to have incorrect calculations and negative values
* All sales should be qty * price
* All prices and sales should be positive
Quantity column appears within the norms

For price
* price can be calculated with sales / qty
* if price is negative in value then *(-1) 

In [0]:
df = df.withColumn("sls_price", 
            F.when(
                (df.sls_sales >= 0) & 
                (df.sls_quantity > 0) & 
                (df.sls_price.isNull())
                , (df.sls_sales/df.sls_quantity)
            )
            .when(
                df.sls_price < 0,
                df.sls_price*(-1)
            )
            .otherwise(df.sls_price))
df.display()

For sales
* with corrected price, calculate sales using qty * price

In [0]:
df = df.withColumn("sls_sales", df.sls_price*df.sls_quantity)
df.display()

## Date transformations

In [0]:
test_dates = df.groupby("sls_order_dt").count()
test_dates.display()

In [0]:
test_dates_v1 = df.select(
    F.max(df.sls_order_dt).alias("max date"), 
    F.min(df.sls_order_dt).alias("min date"), 
    F.round(F.avg(F.length(df.sls_order_dt)), 5).alias("avg date")
)
test_dates_v1.display()

In [0]:
test_dates_v2 = df.where(length(df.sls_order_dt) != 8)
test_dates_v2.display()

Dates provided are string type and there are some dates which have length less than 8 characters.
*  if date is of correct size (8 characters) - modify it with .to_date function
*  otherwise return None

In [0]:
df = df.withColumn("sls_order_dt", 
                   F.when(
                          length(df.sls_order_dt) == 8, F.to_date("sls_order_dt", "yyyyMMdd")
                   ).otherwise(None))
df = df.withColumn("sls_ship_dt", 
                   F.when(
                          length(df.sls_ship_dt) == 8, F.to_date("sls_ship_dt", "yyyyMMdd")
                     ).otherwise(None))
df = df.withColumn("sls_due_dt", 
                   F.when(
                          length(df.sls_due_dt) == 8, F.to_date("sls_due_dt", "yyyyMMdd")
                     ).otherwise(None))
df.display()

In [0]:
df.limit(100).display()

# Write table silver.sls_details

Everything looks ok, so save DataFrame into Delta Table

In [0]:
df.write.mode("overwrite").option("overwriteSchema", True).format("delta").saveAsTable("db_project.silver.crm_sls_details")
df.printSchema()